# Yelp Dataset — Feature Engineering

**Goal**: Engineer the 6 core LOO numerical features (identical to Amazon)
and clean the 3 categorical columns for downstream OHE/embedding experiments.

**Feature set:**
| Type | Features |
|------|---------|
| Numerical (LOO) | user_avg_rating, user_rating_count, business_avg_rating, business_rating_count, business_rating_std, days_since_review |
| Categorical (raw) | city, state, categories |
| Key columns (not features) | user_id, business_id |
| Target | stars |

**Why same 6 numerical features as Amazon?**
Keeps the numerical feature set identical across datasets for a clean
cross-dataset comparison. The categorical columns are what makes Yelp unique.

**Known limitations (same as Amazon):**
- LOO leakage for std (minor, accepted)
- days_since_review uses TRAIN_MAX_DATE as anchor (minor, train-only)

**Output files**
```
yelp/data/
  train_features.csv
  val_features.csv
  test_features.csv
```


## Roadmap
```
STEP 1  — Imports & paths
STEP 2  — Load splits
STEP 3  — Parse timestamps & days_since_review
STEP 4  — User features (LOO)
STEP 5  — Business features (LOO)
STEP 6  — Clean categorical columns
STEP 7  — Apply features to val & test
STEP 8  — Cold start handling
STEP 9  — Final feature overview
STEP 10 — Save enriched CSVs
STEP 11 — Sanity checks
```


---
## Step 1 — Imports & Paths

In [ ]:
import os
import numpy  as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')


In [ ]:
BASE_DIR = r"C:\\Users\\Shivanshu Sirohi\\OneDrive\\Desktop\\Shivanshu\\White Paper\\Personalization\\yelp"
DATA_DIR = r"C:\\Users\\Shivanshu Sirohi\\OneDrive\\Desktop\\Shivanshu\\White Paper\\Personalization\\yelp\\data"
print(f"Data directory: {DATA_DIR}")


---
## Step 2 — Load Splits

In [ ]:
df_train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
df_val   = pd.read_csv(os.path.join(DATA_DIR, 'val.csv'))
df_test  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

TARGET      = 'stars'
CAT_CITY    = 'city'
CAT_STATE   = 'state'
CAT_CATS    = 'categories'

NUM_COLS = [
    'user_avg_rating',    'user_rating_count',
    'business_avg_rating','business_rating_count', 'business_rating_std',
    'days_since_review'
]

print(f"Train : {df_train.shape}")
print(f"Val   : {df_val.shape}")
print(f"Test  : {df_test.shape}")
print(f"Columns: {list(df_train.columns)}")


---
## Step 3 — Parse Timestamps & days_since_review

Convert datetime string to datetime object.
Compute days_since_review relative to TRAIN_MAX_DATE.


In [ ]:
for df_ in [df_train, df_val, df_test]:
    df_['datetime'] = pd.to_datetime(df_['datetime'], errors='coerce')

TRAIN_MAX_DATE = df_train['datetime'].max()

print(f"Train range : {df_train['datetime'].min()} -> {df_train['datetime'].max()}")
print(f"Val range   : {df_val['datetime'].min()} -> {df_val['datetime'].max()}")
print(f"Test range  : {df_test['datetime'].min()} -> {df_test['datetime'].max()}")
print(f"Reference   : {TRAIN_MAX_DATE}")


In [ ]:
# days_since_review — age of review relative to end of train period
for df_ in [df_train, df_val, df_test]:
    df_['days_since_review'] = (TRAIN_MAX_DATE - df_['datetime']).dt.days

print("days_since_review stats (train):")
print(df_train['days_since_review'].describe().round(1))


---
## Step 4 — User Features (Leave-One-Out)

**user_avg_rating**: mean rating this user gives, excluding this row.
**user_rating_count**: how many businesses this user has rated, excluding this row.

LOO formula: (sum_all - this_rating) / (count_all - 1)
np.where used explicitly to set single-review users to NaN
(avoids the clip(lower=1) bug that produces 0.0 instead of NaN).


In [ ]:
# Full user aggregates from train
user_agg = df_train.groupby('user_id')['stars'].agg(
    user_sum   = 'sum',
    user_count = 'count',
    user_std   = 'std',
).reset_index()
user_agg['user_std'] = user_agg['user_std'].fillna(0)

# Global stats — from stats table (each user counted once)
global_avg        = df_train['stars'].mean()
global_user_avg   = user_agg['user_sum'].sum() / user_agg['user_count'].sum()
global_user_count = user_agg['user_count'].mean()

print(f"Global avg rating      : {global_avg:.4f}")
print(f"Global user avg rating : {global_user_avg:.4f}")
print(f"Unique users in train  : {len(user_agg):,}")


In [ ]:
# Merge aggregates onto train and compute LOO
df_train = df_train.merge(user_agg, on='user_id', how='left')

# np.where explicitly produces NaN for single-review users
# This ensures fillna correctly catches them (avoids clip=0/1=0 bug)
loo_denom = df_train['user_count'] - 1
loo_numer = df_train['user_sum']   - df_train['stars']

df_train['user_avg_rating'] = np.where(
    loo_denom > 0,
    loo_numer / loo_denom,
    np.nan
)
df_train['user_rating_count'] = loo_denom.clip(lower=0)

# Fill single-review users with global avg, clip to valid range
df_train['user_avg_rating'] = (df_train['user_avg_rating']
                                .fillna(global_avg)
                                .clip(1.0, 5.0))

df_train.drop(columns=['user_sum','user_count','user_std'], inplace=True)

print(f"user_avg_rating — mean: {df_train['user_avg_rating'].mean():.4f}  "
      f"min: {df_train['user_avg_rating'].min():.4f}  "
      f"max: {df_train['user_avg_rating'].max():.4f}")
print(f"user_rating_count — mean: {df_train['user_rating_count'].mean():.2f}  "
      f"max: {df_train['user_rating_count'].max():.0f}")


---
## Step 5 — Business Features (Leave-One-Out)

Same LOO approach as users.
**business_avg_rating**: mean rating this business receives, excluding this row.
**business_rating_count**: how many users rated this business, excluding this row.
**business_rating_std**: spread of ratings — not strictly LOO (minor leakage accepted).


In [ ]:
# Full business aggregates from train
business_agg = df_train.groupby('business_id')['stars'].agg(
    business_sum   = 'sum',
    business_count = 'count',
    business_std   = 'std',
).reset_index()
business_agg['business_std'] = business_agg['business_std'].fillna(0)

# Global product stats
global_business_avg   = business_agg['business_sum'].sum() / business_agg['business_count'].sum()
global_business_std   = business_agg['business_std'].mean()
global_business_count = business_agg['business_count'].mean()

print(f"Global business avg rating : {global_business_avg:.4f}")
print(f"Global business std        : {global_business_std:.4f}")
print(f"Unique businesses in train : {len(business_agg):,}")


In [ ]:
# Merge and compute LOO
df_train = df_train.merge(business_agg, on='business_id', how='left')

loo_denom = df_train['business_count'] - 1
loo_numer = df_train['business_sum']   - df_train['stars']

df_train['business_avg_rating'] = np.where(
    loo_denom > 0,
    loo_numer / loo_denom,
    np.nan
)
df_train['business_rating_count'] = loo_denom.clip(lower=0)
df_train['business_rating_std']   = df_train['business_std']

df_train['business_avg_rating'] = (df_train['business_avg_rating']
                                    .fillna(global_business_avg)
                                    .clip(1.0, 5.0))
df_train['business_rating_std'] = df_train['business_rating_std'].fillna(0)

df_train.drop(columns=['business_sum','business_count','business_std'],
              inplace=True)

print(f"business_avg_rating — mean: {df_train['business_avg_rating'].mean():.4f}  "
      f"min: {df_train['business_avg_rating'].min():.4f}  "
      f"max: {df_train['business_avg_rating'].max():.4f}")
print(f"business_rating_count — mean: {df_train['business_rating_count'].mean():.2f}  "
      f"max: {df_train['business_rating_count'].max():.0f}")


---
## Step 6 — Clean Categorical Columns

**city**: fill missing with 'Unknown', strip whitespace
**state**: fill missing with 'Unknown', uppercase
**categories**: already normalised to "|" separator in data prep.
  Fill missing with 'Unknown'.

These columns are kept as-is for downstream notebooks:
- Baseline: OHE(state) + OHE(city) + multi-hot(categories)
- Embeddings: learned dense vectors for all three


In [ ]:
for df_ in [df_train, df_val, df_test]:
    df_[CAT_CITY]  = df_[CAT_CITY].fillna('Unknown').str.strip()
    df_[CAT_STATE] = df_[CAT_STATE].fillna('Unknown').str.strip().str.upper()
    df_[CAT_CATS]  = df_[CAT_CATS].fillna('Unknown')

print(f"Unique cities  (train) : {df_train[CAT_CITY].nunique():,}")
print(f"Unique states  (train) : {df_train[CAT_STATE].nunique():,}")
print(f"Unique cat strings (train): {df_train[CAT_CATS].nunique():,}")

# Sample categories
print("\nSample categories:")
for c in df_train[CAT_CATS].dropna().head(5):
    print(f"  {c[:80]}")


---
## Step 7 — Apply Features to Val & Test

Build lookup tables from train stats.
No LOO needed — val/test rows were never in training aggregation.


In [ ]:
# Build lookup tables from stats tables (each user/business appears once)
user_lookup = pd.DataFrame({'user_id': user_agg['user_id']})
user_lookup['user_avg_rating'] = (
    user_agg['user_sum'] / user_agg['user_count']
).clip(1.0, 5.0)
user_lookup['user_rating_count'] = user_agg['user_count']
user_lookup = user_lookup[['user_id','user_avg_rating','user_rating_count']]

business_lookup = pd.DataFrame({'business_id': business_agg['business_id']})
business_lookup['business_avg_rating'] = (
    business_agg['business_sum'] / business_agg['business_count']
).clip(1.0, 5.0)
business_lookup['business_rating_count'] = business_agg['business_count']
business_lookup['business_rating_std']   = business_agg['business_std']
business_lookup = business_lookup[['business_id','business_avg_rating',
                                    'business_rating_count','business_rating_std']]

print(f"User lookup     : {user_lookup.shape}")
print(f"Business lookup : {business_lookup.shape}")


In [ ]:
def apply_features(df, user_lookup, business_lookup):
    df = df.merge(user_lookup,     on='user_id',     how='left')
    df = df.merge(business_lookup, on='business_id', how='left')
    return df

df_val  = apply_features(df_val,  user_lookup, business_lookup)
df_test = apply_features(df_test, user_lookup, business_lookup)

print(f"Val  : {df_val.shape}")
print(f"Test : {df_test.shape}")


---
## Step 8 — Cold Start Handling

Users/businesses in val/test not seen in train → NaN after merge.
Fill with global fallbacks from stats tables (not expanded df).

Min/max filled with 3.0 (neutral midpoint — we know nothing about their range).


In [ ]:
# Global fallbacks from stats tables
global_user_avg_fb     = user_agg['user_sum'].sum() / user_agg['user_count'].sum()
global_business_avg_fb = business_agg['business_sum'].sum() / business_agg['business_count'].sum()
global_business_std_fb = business_agg['business_std'].mean()

print(f"User avg fallback     : {global_user_avg_fb:.4f}")
print(f"Business avg fallback : {global_business_avg_fb:.4f}")
print(f"Business std fallback : {global_business_std_fb:.4f}")


In [ ]:
# Count cold start cases
for name, df_ in [('Val', df_val), ('Test', df_test)]:
    cu = df_['user_avg_rating'].isna().sum()
    cb = df_['business_avg_rating'].isna().sum()
    print(f"{name} — cold start users: {cu:,}  cold start businesses: {cb:,}")


In [ ]:
fills = {
    'user_avg_rating'      : global_user_avg_fb,
    'user_rating_count'    : 0,
    'business_avg_rating'  : global_business_avg_fb,
    'business_rating_count': 0,
    'business_rating_std'  : global_business_std_fb,
}

for df_ in [df_val, df_test]:
    df_.fillna(fills, inplace=True)

# Also fill any train NaNs from LOO edge cases
train_fills = {
    'user_avg_rating'      : global_avg,
    'user_rating_count'    : 0,
    'business_avg_rating'  : global_avg,
    'business_rating_count': 0,
    'business_rating_std'  : 0.0,
}
df_train.fillna(train_fills, inplace=True)

# Verify
for name, df_ in [('Train',df_train),('Val',df_val),('Test',df_test)]:
    nans = df_[NUM_COLS].isna().sum().sum()
    print(f"{name} — NaNs in NUM_COLS: {nans}  {'✓' if nans==0 else '✗ CHECK!'}")


---
## Step 9 — Final Feature Overview

In [ ]:
print(f"Numerical features : {len(NUM_COLS)}")
print(f"Categorical columns: {[CAT_CITY, CAT_STATE, CAT_CATS]}")
print()
print(f"{'Feature':<30} {'Mean':>8} {'Std':>8} {'Min':>8} {'Max':>8}")
print("-" * 60)
for f in NUM_COLS:
    m  = df_train[f].mean()
    s  = df_train[f].std()
    mn = df_train[f].min()
    mx = df_train[f].max()
    print(f"{f:<30} {m:>8.3f} {s:>8.3f} {mn:>8.3f} {mx:>8.3f}")

print()
print(f"Categorical cardinality (train):")
print(f"  city       : {df_train[CAT_CITY].nunique():,} unique values")
print(f"  state      : {df_train[CAT_STATE].nunique():,} unique values")
print(f"  categories : {df_train[CAT_CATS].nunique():,} unique strings")

# Count unique individual category tokens
all_tokens = set()
for c in df_train[CAT_CATS].dropna():
    all_tokens.update(c.split('|'))
print(f"  category tokens : {len(all_tokens):,} unique tokens")


---
## Step 10 — Save Enriched CSVs

In [ ]:
train_out = os.path.join(DATA_DIR, 'train_features.csv')
val_out   = os.path.join(DATA_DIR, 'val_features.csv')
test_out  = os.path.join(DATA_DIR, 'test_features.csv')

df_train.to_csv(train_out, index=False)
df_val.to_csv(val_out,     index=False)
df_test.to_csv(test_out,   index=False)

for fpath in [train_out, val_out, test_out]:
    size_mb = os.path.getsize(fpath) / 1e6
    print(f"  {os.path.basename(fpath):<25}  {size_mb:.1f} MB")


---
## Step 11 — Sanity Checks

In [ ]:
train_check = pd.read_csv(train_out)
val_check   = pd.read_csv(val_out)
test_check  = pd.read_csv(test_out)

print("Reloaded shapes:")
print(f"  train_features : {train_check.shape}")
print(f"  val_features   : {val_check.shape}")
print(f"  test_features  : {test_check.shape}")


In [ ]:
for name, df_ in [('train',train_check),('val',val_check),('test',test_check)]:
    nans = df_[NUM_COLS].isna().sum().sum()
    print(f"  {name} NaNs in numerical features: {nans}  {'✓' if nans==0 else '✗'}")


In [ ]:
print("=" * 60)
print("YELP — FEATURE ENGINEERING COMPLETE")
print("=" * 60)
print(f"  Numerical features (6) :")
for f in NUM_COLS:
    print(f"    {f}")
print()
print(f"  Categorical columns (3) : city, state, categories")
print(f"    city       : {{df_train['city'].nunique():,}} unique values")
print(f"    state      : {{df_train['state'].nunique():,}} unique values")
print(f"    categories : multi-label, '|' separated")
print()
print(f"  Train : {{len(train_check):,}} rows")
print(f"  Val   : {{len(val_check):,}} rows")
print(f"  Test  : {{len(test_check):,}} rows")
print()
print(f"  Next: run baseline model notebooks")
print("=" * 60)
